# Procesamiento Distribuido PySpark y Arquitectura Medallion - Vacunación MINSA
**Curso:** Base de Datos Avanzadas y Big Data (NRC 23811)  
**Objetivo:** Ingesta, aplicación de reglas `TR-01` a `TR-06`, particionamiento Apache Parquet (Snappy) y consultas con *Partition Pruning*.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('MINSA_Medallion_ETL').config('spark.sql.adaptive.enabled', 'true').getOrCreate()
df_silver = spark.read.parquet('../data/silver/vacunacion_parquet')
df_silver.createOrReplaceTempView('vw_silver_vacunacion')

[INFO] SparkSession iniciada: Spark v3.5.1 | AQE=true | skewJoin=true
[INFO] Registros leidos en Zona Bronze: 85,420,118 filas (14.24 GB)
[INFO] Registros limpios en Zona Silver (Parquet): 83,915,402 filas (2.15 GB)


In [2]:
# Consulta Analitica Distribuida con Partition Pruning (Departamento = 15 LIMA, Anio = 2022)
spark.sql('''
    SELECT rango_edad, fabricante, COUNT(*) AS total_dosis, ROUND(AVG(edad_valida), 1) AS edad_promedio
    FROM vw_silver_vacunacion
    WHERE id_departamento = '15' AND anio_vacunacion = 2022 AND dosis = 3
    GROUP BY rango_edad, fabricante
    ORDER BY total_dosis DESC
''').show(5, truncate=False)

+----------+---------------+-----------+-------------+
|rango_edad|fabricante     |total_dosis|edad_promedio|
+----------+---------------+-----------+-------------+
|30-59     |PFIZER         |4128940    |44.2         |
|60+       |PFIZER         |2315412    |71.8         |
|18-29     |ASTRAZENECA    |1489205    |24.1         |
|30-59     |MODERNA        |984110     |46.5         |
|18-29     |PFIZER         |872319     |23.6         |
+----------+---------------+-----------+-------------+
